# Multi-Agent Trip Planner with Human-in-the-Loop Middleware & SQLite Persistence

### Architectural Pattern: **Supervisor (Subagents-as-Tools) + `HumanInTheLoopMiddleware` + `SqliteSaver`**

In this notebook, we build an interactive, multi-stage **Trip Planner Multi-Agent System** where a central **Main Orchestrator Agent** coordinates two specialized sub-agents (**Hotel Sub-Agent** and **Flight Sub-Agent**) while enforcing **Human-in-the-Loop (HITL) approvals** at every decision boundary and persisting all conversation state in **SQLite (`SqliteSaver`)**.

```
                           ┌────────────────────┐
                           │        User        │
                           └─────────┬──────────┘
                                     │ Query / HITL Approvals
                                     ▼
                    ┌──────────────────────────────────┐
                    │        Main Orchestrator         │  ◄─── SqliteSaver (trip_planner_memory.db)
                    │  + HumanInTheLoopMiddleware      │  ◄─── Interrupts on Plan, Hotel & Flight Gates
                    └────────┬───────────────┬─────────┘
                             │               │
            ┌────────────────┘               └────────────────┐
            ▼                                                 ▼
┌────────────────────────┐                       ┌────────────────────────┐
│    Hotel Sub-Agent     │                       │    Flight Sub-Agent    │
│  • search_hotels       │                       │  • search_flights      │
│  • book_hotel          │                       │  • book_flight         │
└────────────────────────┘                       └────────────────────────┘
```

### The 5-Stage Interactive Workflow:
1. **Stage 1 — Capture Basic User Info:** The Main Agent collects **User Name**, **Total Budget**, **Source City**, and **Destination City**.
2. **Stage 2 — Plan Approval (`HITL Gate #1`):** Once all 4 details are captured, the Main Agent notifies the user and invokes `find_hotels_subagent`. **`HumanInTheLoopMiddleware` intercepts the call** so the user can **Approve** or **Edit** the trip plan (e.g., updating the budget) before spinning up the **Hotel Sub-Agent**.
3. **Stage 3 — Hotel Search & Booking Approval (`HITL Gate #2`):** The Hotel Sub-Agent returns available hotels. The Main Agent displays the hotel details and invokes `book_hotel_subagent`. **`HumanInTheLoopMiddleware` pauses execution BEFORE booking** so the user can **Approve** or **Edit** the selected hotel.
4. **Stage 4 — Flight Dates, Search & Booking Approval (`HITL Gate #3`):** After hotel booking confirmation, the Main Agent asks the user for their **Flight Travel Date**. Once provided, it calls `search_flights_subagent` to show flight options and invokes `book_flight_subagent`. **`HumanInTheLoopMiddleware` pauses execution BEFORE booking** for user approval.
5. **Stage 5 — Final Confirmation:** The Main Agent compiles the confirmed Hotel Reference, Flight PNR, and Budget summary and reverts to the user.

## 1. Environment Setup & Persistent SQLite Checkpointer (`SqliteSaver`)
Unlike `InMemorySaver` (which loses chat history when the Python process restarts), **`SqliteSaver`** persists every checkpoint, message, and pending HITL interrupt directly into a local SQLite database file (`trip_planner_memory.db`).

In [1]:
import os
import sqlite3
from typing import Any, Dict, List
from dotenv import load_dotenv

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware.human_in_the_loop import HumanInTheLoopMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command

# Load API keys from .env
load_dotenv("../.env")
load_dotenv()

# Create a persistent SQLite connection and wrap it in LangGraph's SqliteSaver
DB_PATH = "trip_planner_memory.db"
sqlite_conn = sqlite3.connect(DB_PATH, check_same_thread=False)
sqlite_checkpointer = SqliteSaver(sqlite_conn)

print(f"✓ Environment loaded and SqliteSaver connected to: {DB_PATH}")

✓ Environment loaded and SqliteSaver connected to: trip_planner_memory.db


## 2. Mock Databases & Domain Tools (`Hotel` & `Flight`)
Each sub-agent has its own specialized set of tools:
- **Hotel Tools:** `search_hotels` (read-only search) and `book_hotel` (state-mutating reservation).
- **Flight Tools:** `search_flights` (read-only search by date) and `book_flight` (state-mutating ticket issuance).

In [2]:
# ==================== MOCK DATABASES ====================
HOTEL_DB: Dict[str, List[Dict[str, Any]]] = {
    "goa": [
        {"hotel_id": "H-GOA-01", "name": "Taj Exotica Resort & Spa Goa", "price_per_night": 220.0, "rating": 4.9, "amenities": "Private Beach, Pool, Breakfast Included, Spa"},
        {"hotel_id": "H-GOA-02", "name": "Lemon Tree Amarante Beach Resort", "price_per_night": 110.0, "rating": 4.5, "amenities": "Near Candolim Beach, Pool, Free Wi-Fi, Breakfast"},
        {"hotel_id": "H-GOA-03", "name": "Zostel Goa Calangute", "price_per_night": 45.0, "rating": 4.2, "amenities": "Backpacker Hostel, Cafe, Common Lounge, Free Wi-Fi"}
    ],
    "tokyo": [
        {"hotel_id": "H-TYO-01", "name": "Park Hyatt Shinjuku Tokyo", "price_per_night": 450.0, "rating": 4.9, "amenities": "Skyline Views, Luxury Spa, Fine Dining, Indoor Pool"},
        {"hotel_id": "H-TYO-02", "name": "Hotel Sunroute Plaza Shinjuku", "price_per_night": 140.0, "rating": 4.5, "amenities": "2 mins to Shinjuku Station, Free Wi-Fi, Buffet Breakfast"},
        {"hotel_id": "H-TYO-03", "name": "Shibuya Capsule Inn", "price_per_night": 65.0, "rating": 4.1, "amenities": "Modern Pods, Lockers, Lounge, Central Shibuya"}
    ]
}

FLIGHT_DB: List[Dict[str, Any]] = [
    {"flight_number": "AI-805", "airline": "Air India", "departure_time": "08:30 AM", "arrival_time": "11:15 AM", "class": "Economy", "price": 180.0},
    {"flight_number": "6E-2042", "airline": "IndiGo", "departure_time": "01:45 PM", "arrival_time": "04:20 PM", "class": "Economy Saver", "price": 135.0},
    {"flight_number": "UK-981", "airline": "Vistara Premier", "departure_time": "07:00 PM", "arrival_time": "09:45 PM", "class": "Premium Economy", "price": 240.0}
]

BOOKINGS_DB: Dict[str, List[Dict[str, Any]]] = {"hotels": [], "flights": []}


# ==================== HOTEL SUB-AGENT TOOLS ====================
@tool
def search_hotels(city: str, max_total_budget: float) -> str:
    """Search available hotels in the destination city and compare against the user's total trip budget."""
    city_key = city.lower().strip()
    hotels = HOTEL_DB.get(city_key, HOTEL_DB["goa"])
    lines = [f"Available Hotels in {city.title()} (Total Trip Budget: ${max_total_budget:.2f}):"]
    for h in hotels:
        est_3_nights = h["price_per_night"] * 3
        lines.append(
            f"- [{h['hotel_id']}] {h['name']} | ${h['price_per_night']:.2f}/night "
            f"(3 nights: ${est_3_nights:.2f}) | Rating: {h['rating']}★ | Amenities: {h['amenities']}"
        )
    return "\n".join(lines)


@tool
def book_hotel(guest_name: str, city: str, hotel_name: str, price_per_night: float, nights: int = 3) -> str:
    """Book a specific hotel in the destination city for the guest and generate a confirmation reference."""
    total_cost = round(price_per_night * nights, 2)
    booking_ref = f"HTL-{abs(hash(f'{guest_name}-{hotel_name}-{city}')) % 90000 + 10000}"
    BOOKINGS_DB["hotels"].append({
        "booking_ref": booking_ref, "guest_name": guest_name, "city": city.title(),
        "hotel_name": hotel_name, "nights": nights, "total_cost": total_cost
    })
    return (
        f"HOTEL BOOKING CONFIRMED!\n"
        f"- Booking Reference: {booking_ref}\n"
        f"- Guest: {guest_name} | Hotel: {hotel_name} ({city.title()})\n"
        f"- Stay: {nights} nights @ ${price_per_night:.2f}/night | Total Cost: ${total_cost:.2f}"
    )


# ==================== FLIGHT SUB-AGENT TOOLS ====================
@tool
def search_flights(source: str, destination: str, travel_date: str) -> str:
    """Search available flights between source and destination on a specific travel_date."""
    lines = [f"Available Flights from {source.title()} to {destination.title()} on {travel_date}:"]
    for f in FLIGHT_DB:
        lines.append(
            f"- Flight {f['flight_number']} ({f['airline']}) | "
            f"Departs: {f['departure_time']} -> Arrives: {f['arrival_time']} | "
            f"Class: {f['class']} | Fare: ${f['price']:.2f}"
        )
    return "\n".join(lines)


@tool
def book_flight(passenger_name: str, source: str, destination: str, travel_date: str, flight_number: str, airline: str, price: float) -> str:
    """Book a specific flight ticket for the passenger and generate a confirmed PNR."""
    pnr = f"FL-PNR-{abs(hash(f'{passenger_name}-{flight_number}-{travel_date}')) % 90000 + 10000}"
    BOOKINGS_DB["flights"].append({
        "pnr": pnr, "passenger_name": passenger_name, "route": f"{source.title()} -> {destination.title()}",
        "travel_date": travel_date, "flight_number": flight_number, "airline": airline, "price": price
    })
    return (
        f"FLIGHT TICKET CONFIRMED!\n"
        f"- PNR Number: {pnr}\n"
        f"- Passenger: {passenger_name} | Route: {source.title()} -> {destination.title()}\n"
        f"- Date: {travel_date} | Flight: {flight_number} ({airline}) | Fare: ${price:.2f}"
    )

print("✓ Hotel and Flight databases & domain tools initialized.")

✓ Hotel and Flight databases & domain tools initialized.


## 3. Build the Specialized Sub-Agents (`hotel_subagent` & `flight_subagent`)
Each sub-agent is an independent agent created with `create_agent`. Notice how neither sub-agent talks directly to the user—they receive tasks from the Main Agent and return their results back to the Main Agent.

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# 1. Hotel Sub-Agent
hotel_subagent = create_agent(
    model=llm,
    tools=[search_hotels, book_hotel],
    system_prompt=(
        "You are the specialized Hotel Sub-Agent. "
        "When asked to search hotels, call `search_hotels` and return ALL options with exact prices and ratings. "
        "When asked to book a hotel, call `book_hotel` and return the confirmation reference."
    ),
)

# 2. Flight Sub-Agent
flight_subagent = create_agent(
    model=llm,
    tools=[search_flights, book_flight],
    system_prompt=(
        "You are the specialized Flight Sub-Agent. "
        "When asked to search flights, call `search_flights` and return ALL flight numbers, timings, and fares. "
        "When asked to book a flight, call `book_flight` and return the confirmed PNR."
    ),
)

def _extract_subagent_text(response: dict) -> str:
    """Helper to extract clean text from sub-agent response messages."""
    last_msg = response["messages"][-1]
    if isinstance(last_msg.content, list):
        return "\n".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in last_msg.content)
    return str(last_msg.content)

print("✓ Specialized sub-agents (hotel_subagent, flight_subagent) created.")

✓ Specialized sub-agents (hotel_subagent, flight_subagent) created.


## 4. Sub-Agent Delegation Tools & `HumanInTheLoopMiddleware` Configuration
To enforce **`HumanInTheLoopMiddleware` for every single approval mechanism**, we expose 4 task-specific delegation tools to the Main Orchestrator Agent and configure `interrupt_on` for the **3 approval gates**:

| Stage | Delegation Tool | HITL Middleware Gate? | Allowed Decisions | Purpose |
| :--- | :--- | :--- | :--- | :--- |
| **Gate #1: Plan Approval** | `find_hotels_subagent` | **YES (`interrupt_on`)** | `approve`, `edit`, `reject` | Pauses after capturing Name, Budget, Source, Destination so user approves/edits the plan before spinning up Hotel Finder. |
| **Gate #2: Hotel Booking** | `book_hotel_subagent` | **YES (`interrupt_on`)** | `approve`, `edit`, `reject` | Pauses after showing hotel options so user approves/edits the hotel before booking. |
| **Flight Search (Read-Only)** | `search_flights_subagent` | No (runs immediately) | — | Runs once user provides Flight Dates to fetch and show available flights. |
| **Gate #3: Flight Booking** | `book_flight_subagent` | **YES (`interrupt_on`)** | `approve`, `edit`, `reject` | Pauses after showing flight details so user approves/edits the flight before booking. |

In [4]:
@tool
def find_hotels_subagent(user_name: str, source: str, destination: str, total_budget: float) -> str:
    """[HITL Gate #1: Plan Approval] Call this once user_name, source, destination, and total_budget
    are captured. After user approves/edits the plan, spins up the Hotel Sub-Agent to search hotels."""
    prompt = f"Search for available hotels in {destination} for {user_name} (budget: ${total_budget})."
    return _extract_subagent_text(hotel_subagent.invoke({"messages": [{"role": "user", "content": prompt}]}))


@tool
def book_hotel_subagent(
    user_name: str,
    destination: str,
    hotel_name: str,
    price_per_night: float,
    nights: int,
    available_hotels_summary: str,
) -> str:
    """[HITL Gate #2: Hotel Booking Approval] Call this immediately after `find_hotels_subagent` returns
    options. Pass the recommended hotel and `available_hotels_summary` so the user can approve or edit before booking."""
    prompt = f"Book hotel '{hotel_name}' in {destination} for guest '{user_name}' for {nights} nights at ${price_per_night}/night."
    return _extract_subagent_text(hotel_subagent.invoke({"messages": [{"role": "user", "content": prompt}]}))


@tool
def search_flights_subagent(source: str, destination: str, travel_date: str) -> str:
    """Call this once the hotel is booked AND the user has provided their flight `travel_date`.
    Spins up the Flight Sub-Agent to search available flights and return their details."""
    prompt = f"Search available flights from {source} to {destination} on {travel_date}."
    return _extract_subagent_text(flight_subagent.invoke({"messages": [{"role": "user", "content": prompt}]}))


@tool
def book_flight_subagent(
    passenger_name: str,
    source: str,
    destination: str,
    travel_date: str,
    flight_number: str,
    airline: str,
    price: float,
    available_flights_summary: str,
) -> str:
    """[HITL Gate #3: Flight Booking Approval] Call this immediately after `search_flights_subagent` returns
    flight options. Pass the recommended flight and `available_flights_summary` so the user can approve or edit before booking."""
    prompt = (
        f"Book flight {flight_number} ({airline}) from {source} to {destination} "
        f"on {travel_date} for passenger '{passenger_name}' at fare ${price}."
    )
    return _extract_subagent_text(flight_subagent.invoke({"messages": [{"role": "user", "content": prompt}]}))


# Configure HumanInTheLoopMiddleware for all 3 approval gates
hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "find_hotels_subagent": {"allowed_decisions": ["approve", "edit", "reject"]},
        "book_hotel_subagent": {"allowed_decisions": ["approve", "edit", "reject"]},
        "book_flight_subagent": {"allowed_decisions": ["approve", "edit", "reject"]},
    },
    description_prefix="TRIP PLANNER HITL APPROVAL GATE",
)

MAIN_ORCHESTRATOR_PROMPT = """You are the Main Trip Planner Orchestrator Agent.
Follow this strict 5-stage workflow:

1. CAPTURE BASIC INFO: Ensure you have `user_name`, `total_budget`, `source`, and `destination`.
   If any are missing, ask the user for them and do NOT call any tools.
2. PLAN APPROVAL (HITL Gate #1): Once all 4 details are known, notify the user in your message that you have all basic info and call `find_hotels_subagent` so the user can approve/edit the plan.
3. HOTEL SELECTION & BOOKING APPROVAL (HITL Gate #2): When `find_hotels_subagent` returns the hotels, list ALL hotel options with prices/amenities in your message AND in the same turn call `book_hotel_subagent` with your recommended hotel so the middleware pauses for user approval before booking.
4. FLIGHT DATES & FLIGHT BOOKING APPROVAL (HITL Gate #3): Once `book_hotel_subagent` confirms the hotel booking, show the confirmation ID and ask the user for their preferred **Flight Travel Date** (if not yet given).
   When the user gives the flight date, call `search_flights_subagent`. Once it returns the flights, display ALL flight details in your message AND in the same turn call `book_flight_subagent` with the best flight so the middleware pauses for user approval before booking.
5. FINAL CONFIRMATION: Once `book_flight_subagent` confirms the PNR, present the complete confirmed Trip Itinerary (Traveler, Route, Hotel Ref, Flight PNR, Total Spent vs Budget).
"""

main_trip_planner = create_agent(
    model=llm,
    tools=[find_hotels_subagent, book_hotel_subagent, search_flights_subagent, book_flight_subagent],
    system_prompt=MAIN_ORCHESTRATOR_PROMPT,
    middleware=[hitl_middleware],
    checkpointer=sqlite_checkpointer,
)

print("✓ Main Orchestrator Agent compiled with HumanInTheLoopMiddleware and SqliteSaver!")

✓ Main Orchestrator Agent compiled with HumanInTheLoopMiddleware and SqliteSaver!


## 5. End-to-End Execution Walkthrough Across All 5 Stages
First, let's define a display helper so we can clearly inspect:
- What the **Main Agent** says to the user,
- What the **Sub-Agents** returned behind the scenes, and
- What **`HumanInTheLoopMiddleware` interrupt** is currently waiting for user approval.

In [5]:
import json

def inspect_step(result: Dict[str, Any], step_title: str):
    print("=" * 88)
    print(f"🔹 {step_title}")
    print("=" * 88)
    
    messages = result.get("messages", [])
    # Print latest ToolMessage if a subagent just ran
    for m in reversed(messages):
        if isinstance(m, ToolMessage):
            print(f"\n[Sub-Agent Tool Output ({m.name})]:\n{m.content}")
            break

    # Print latest AIMessage text
    for m in reversed(messages):
        if isinstance(m, AIMessage):
            text = _extract_subagent_text({"messages": [m]}).strip()
            if text:
                print(f"\n[Main Orchestrator Agent]:\n{text}")
            break

    # Check if HumanInTheLoopMiddleware triggered an interrupt
    interrupts = result.get("__interrupt__", [])
    if interrupts:
        payload = interrupts[0].value
        act = payload["action_requests"][0]
        rev = payload["review_configs"][0]
        print("\n⏸️  [HUMAN-IN-THE-LOOP MIDDLEWARE INTERRUPT TRIGGERED!]")
        print(f"   • Paused Before Tool : {act['name']}")
        print(f"   • Allowed Decisions  : {rev['allowed_decisions']}")
        print(f"   • Proposed Arguments : {json.dumps(act['args'], indent=6)}")
    else:
        print("\n✅ [Status]: Waiting for next user chat message (No pending HITL interrupt).")
    print("=" * 88 + "\n")

print("✓ Inspection helper `inspect_step` ready.")

✓ Inspection helper `inspect_step` ready.


### Stage 1: Capturing Basic User Info & Triggering HITL Gate #1 (Plan Approval)
- **Turn 1A:** User says *"Hi, I'm Sachin and I want to plan a trip to Goa."* (Missing `source` and `total_budget`). The Main Agent asks for the missing info.
- **Turn 1B:** User provides `source="Delhi"` and `total_budget=1500`. Now all 4 fields are captured! The Main Agent notifies the user and calls `find_hotels_subagent`, which is immediately **paused by `HumanInTheLoopMiddleware` (Gate #1: Plan Approval)**!

In [6]:
config = {"configurable": {"thread_id": "trip-demo-thread-101"}}

# Turn 1A: User provides only Name and Destination
res_1a = main_trip_planner.invoke(
    {"messages": [{"role": "user", "content": "Hi, I am Sachin and I want to plan a trip to Goa."}]},
    config=config,
)
inspect_step(res_1a, "Stage 1A: Partial User Info (Missing Source & Budget)")

# Turn 1B: User provides Source City and Budget -> Triggers HITL Gate #1 (Plan Approval)
res_1b = main_trip_planner.invoke(
    {"messages": [{"role": "user", "content": "I will be flying from Delhi and my total budget is $1500."}]},
    config=config,
)
inspect_step(res_1b, "Stage 1B -> Stage 2: All Basic Info Captured -> HITL Gate #1 (Plan Approval)")

🔹 Stage 1A: Partial User Info (Missing Source & Budget)

[Main Orchestrator Agent]:
Hello Sachin! I would be delighted to help you plan your trip to Goa. Before we proceed, could you please share:
1. Your **Source City** (where you will be traveling from)
2. Your **Total Trip Budget** (in USD)

✅ [Status]: Waiting for next user chat message (No pending HITL interrupt).

🔹 Stage 1B -> Stage 2: All Basic Info Captured -> HITL Gate #1 (Plan Approval)

[Main Orchestrator Agent]:
Thank you, Sachin! I have captured all your basic trip details:
- **Traveler Name:** Sachin
- **Route:** Delhi -> Goa
- **Total Budget:** $1,500.00

I am now submitting this Trip Plan for your approval before spinning up the Hotel Finder Sub-Agent.

⏸️  [HUMAN-IN-THE-LOOP MIDDLEWARE INTERRUPT TRIGGERED!]
   • Paused Before Tool : find_hotels_subagent
   • Allowed Decisions  : ['approve', 'edit', 'reject']
   • Proposed Arguments : {
      "user_name": "Sachin",
      "source": "Delhi",
      "destination": "Goa",
 

### Stage 2 $\rightarrow$ Stage 3: User Approves/Edits the Plan (`HITL Gate #1`) $\rightarrow$ Hotel Sub-Agent Searches $\rightarrow$ Pauses at `HITL Gate #2` (Hotel Booking Approval)
Here, the user decides to **Edit** the plan during Approval Gate #1 (increasing `total_budget` from `$1500` to `$1800`) using `Command(resume={"decisions": [{"type": "edit", ...}]})`.
- As soon as the decision is submitted, `find_hotels_subagent` spins up the **Hotel Sub-Agent**, which searches hotels in Goa.
- The Main Agent then displays all 3 hotel options and invokes `book_hotel_subagent` for the recommended hotel, which **automatically pauses at HITL Gate #2** so the user can approve the hotel before booking!

In [7]:
# User approves the plan with an edit (updating total_budget to $1800)
res_2 = main_trip_planner.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "find_hotels_subagent",
                        "args": {
                            "user_name": "Sachin",
                            "source": "Delhi",
                            "destination": "Goa",
                            "total_budget": 1800.0,
                        },
                    },
                }
            ]
        }
    ),
    config=config,
)
inspect_step(res_2, "Stage 3: Plan Approved/Edited -> Hotel Sub-Agent Search -> HITL Gate #2 (Hotel Booking Approval)")

🔹 Stage 3: Plan Approved/Edited -> Hotel Sub-Agent Search -> HITL Gate #2 (Hotel Booking Approval)

[Sub-Agent Tool Output (find_hotels_subagent)]:
Available Hotels in Goa (Total Trip Budget: $1800.00):
- [H-GOA-01] Taj Exotica Resort & Spa Goa | $220.00/night (3 nights: $660.00) | Rating: 4.9★ | Amenities: Private Beach, Pool, Breakfast Included, Spa
- [H-GOA-02] Lemon Tree Amarante Beach Resort | $110.00/night (3 nights: $330.00) | Rating: 4.5★ | Amenities: Near Candolim Beach, Pool, Free Wi-Fi, Breakfast
- [H-GOA-03] Zostel Goa Calangute | $45.00/night (3 nights: $135.00) | Rating: 4.2★ | Amenities: Backpacker Hostel, Cafe, Common Lounge, Free Wi-Fi

[Main Orchestrator Agent]:
Here are the available hotels in Goa found by our Hotel Sub-Agent:
1. **Taj Exotica Resort & Spa Goa** — $220.00/night (3 nights: $660.00) | 4.9★ | Private Beach, Pool, Breakfast Included, Spa
2. **Lemon Tree Amarante Beach Resort** — $110.00/night (3 nights: $330.00) | 4.5★ | Near Candolim Beach, Pool, Free W

### Stage 3 $\rightarrow$ Stage 4A: User Approves Hotel Booking (`HITL Gate #2`) $\rightarrow$ Main Agent Confirms Hotel & Asks for Flight Dates
Now the user approves the hotel booking (`{"type": "approve"}`).
- The **Hotel Sub-Agent** books **Lemon Tree Amarante Beach Resort** and returns Booking Reference `HTL-13184`.
- Before searching or booking flights, the Main Agent notifies the user of the hotel confirmation and **asks for the user's preferred Flight Travel Date**.

In [8]:
# User approves the Hotel Booking at Gate #2
res_3 = main_trip_planner.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
)
inspect_step(res_3, "Stage 4A: Hotel Booked -> Main Agent Asks User for Flight Travel Date")

🔹 Stage 4A: Hotel Booked -> Main Agent Asks User for Flight Travel Date

[Sub-Agent Tool Output (book_hotel_subagent)]:
HOTEL BOOKING CONFIRMED!
- Booking Reference: HTL-13184
- Guest: Sachin | Hotel: Lemon Tree Amarante Beach Resort (Goa)
- Stay: 3 nights @ $110.00/night | Total Cost: $330.00

[Main Orchestrator Agent]:
Your hotel booking at **Lemon Tree Amarante Beach Resort** in Goa for 3 nights ($330.00 total) is confirmed! Your booking reference is **HTL-13184**.

Next, to search and book your flight from **Delhi to Goa**, please share your preferred **Flight Travel Date** (e.g., YYYY-MM-DD).

✅ [Status]: Waiting for next user chat message (No pending HITL interrupt).


### Stage 4B: User Provides Flight Date $\rightarrow$ Flight Sub-Agent Searches & Shows Details $\rightarrow$ Pauses at `HITL Gate #3` (Flight Booking Approval)
- The user provides their travel date (`"2026-11-15"`).
- The Main Agent calls `search_flights_subagent` to retrieve all flights from Delhi to Goa on `2026-11-15`.
- It displays the flight options and calls `book_flight_subagent`, which **pauses at `HITL Gate #3`** so the user can review the flight details and approve before booking!

In [9]:
# User provides preferred Flight Travel Date
res_4 = main_trip_planner.invoke(
    {"messages": [{"role": "user", "content": "My flight travel date is 2026-11-15."}]},
    config=config,
)
inspect_step(res_4, "Stage 4B: Flight Sub-Agent Search & Details -> HITL Gate #3 (Flight Booking Approval)")

🔹 Stage 4B: Flight Sub-Agent Search & Details -> HITL Gate #3 (Flight Booking Approval)

[Sub-Agent Tool Output (search_flights_subagent)]:
Available Flights from Delhi to Goa on 2026-11-15:
- Flight AI-805 (Air India) | Departs: 08:30 AM -> Arrives: 11:15 AM | Class: Economy | Fare: $180.00
- Flight 6E-2042 (IndiGo) | Departs: 01:45 PM -> Arrives: 04:20 PM | Class: Economy Saver | Fare: $135.00
- Flight UK-981 (Vistara Premier) | Departs: 07:00 PM -> Arrives: 09:45 PM | Class: Premium Economy | Fare: $240.00

[Main Orchestrator Agent]:
Here are the available flights from **Delhi to Goa** on **2026-11-15**:
1. **Flight AI-805 (Air India)** — Departs: 08:30 AM -> Arrives: 11:15 AM | Economy | **$180.00**
2. **Flight 6E-2042 (IndiGo)** — Departs: 01:45 PM -> Arrives: 04:20 PM | Economy Saver | **$135.00**
3. **Flight UK-981 (Vistara Premier)** — Departs: 07:00 PM -> Arrives: 09:45 PM | Premium Economy | **$240.00**

I have selected **Flight 6E-2042 (IndiGo)** at **$135.00** for your appr

### Stage 5: User Approves Flight Booking (`HITL Gate #3`) $\rightarrow$ Final Confirmed Itinerary & SQLite Verification
- The user approves the flight booking (`Command(resume={"decisions": [{"type": "approve"}]})`).
- The **Flight Sub-Agent** issues the ticket (`FL-PNR-17130`), and the **Main Orchestrator Agent** reverts to the user with the complete confirmed itinerary!

In [10]:
# User approves the Flight Booking at Gate #3
res_5 = main_trip_planner.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
)
inspect_step(res_5, "Stage 5: Flight Booked -> Final Trip Confirmation by Main Agent")

# Verify SQLite database persistence on disk
cursor = sqlite_conn.cursor()
count = cursor.execute(
    "SELECT COUNT(*) FROM checkpoints WHERE thread_id = ?",
    ("trip-demo-thread-101",),
).fetchone()[0]
print(f"📦 SQLite Persistence Check (`{DB_PATH}`):")
print(f"   • Persisted Checkpoints for thread 'trip-demo-thread-101': {count} checkpoints saved on disk!")

🔹 Stage 5: Flight Booked -> Final Trip Confirmation by Main Agent

[Sub-Agent Tool Output (book_flight_subagent)]:
FLIGHT TICKET CONFIRMED!
- PNR Number: FL-PNR-17130
- Passenger: Sachin | Route: Delhi -> Goa
- Date: 2026-11-15 | Flight: 6E-2042 (IndiGo) | Fare: $135.00

[Main Orchestrator Agent]:
🎉 **Your Complete Trip to Goa is Confirmed!**

### 1. Traveler & Route Overview
- **Traveler Name:** Sachin
- **Route:** Delhi -> Goa
- **Approved Trip Budget:** $1,800.00

### 2. Confirmed Hotel Booking
- **Hotel:** Lemon Tree Amarante Beach Resort (Goa)
- **Duration:** 3 Nights @ $110.00/night
- **Total Hotel Cost:** $330.00
- **Booking Reference:** `HTL-13184`

### 3. Confirmed Flight Ticket
- **Flight:** 6E-2042 (IndiGo)
- **Travel Date:** 2026-11-15 (Departs 01:45 PM -> Arrives 04:20 PM)
- **Ticket Fare:** $135.00
- **PNR Number:** `FL-PNR-17130`

### 4. Budget Summary
- **Total Spent (Hotel + Flight):** $465.00
- **Remaining Budget for Activities & Dining:** $1,335.00

Have a wonderful 